# Bhagavata morphology fine-tune (free Colab)

**Setup:** Runtime -> Change runtime type -> **T4 GPU**

**Fast path:** run the next cell, then upload these two files from your machine:
- `training/data/morphology_n8000_train.jsonl`
- `training/data/morphology_n8000_val.jsonl`

Training takes about **15-30 minutes** on a free T4.

In [ ]:
!nvidia-smi
# Colab ships an old torchao; peft rejects it unless upgraded or removed.
!pip -q uninstall -y torchao
!pip -q install "transformers>=4.44" "datasets>=2.19" "accelerate>=0.33" "peft>=0.12" "trl>=0.9.6"

In [ ]:
from pathlib import Path
from google.colab import files

data_dir = Path('/content/training_data')
data_dir.mkdir(exist_ok=True)

train_path = data_dir / 'morphology_n8000_train.jsonl'
val_path = data_dir / 'morphology_n8000_val.jsonl'

if not (train_path.exists() and val_path.exists()):
    print('Upload train + val JSONL files')
    uploaded = files.upload()
    for name, content in uploaded.items():
        target = data_dir / Path(name).name
        target.write_bytes(content)
        print('saved', target)

assert train_path.exists() and val_path.exists(), 'Missing JSONL files'
print('ready:', train_path, val_path)

In [ ]:
import json
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import SFTTrainer

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
OUTPUT_DIR = '/content/bhagavata-morph-lora'

def load_jsonl(path):
    rows = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

def to_chat(rows, tokenizer):
    out = []
    for row in rows:
        messages = [
            {'role': 'system', 'content': row['instruction']},
            {'role': 'user', 'content': row['input']},
            {'role': 'assistant', 'content': row['output']},
        ]
        out.append({'text': tokenizer.apply_chat_template(messages, tokenize=False)})
    return out

train_rows = load_jsonl(train_path)
val_rows = load_jsonl(val_path)
print(f'train={len(train_rows)} val={len(val_rows)}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=dtype, device_map='auto')
model = get_peft_model(model, LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, task_type='CAUSAL_LM', target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj']))
model.print_trainable_parameters()

trainer = SFTTrainer(
    model=model,
    train_dataset=Dataset.from_list(to_chat(train_rows, tokenizer)),
    eval_dataset=Dataset.from_list(to_chat(val_rows, tokenizer)),
    processing_class=tokenizer,
    max_seq_length=768,
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=1,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=16,
        learning_rate=2e-4,
        logging_steps=25,
        eval_strategy='steps',
        eval_steps=100,
        save_steps=100,
        save_total_limit=1,
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        report_to='none',
    ),
)
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('saved to', OUTPUT_DIR)

In [ ]:
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=dtype, device_map='auto')
model = PeftModel.from_pretrained(base, OUTPUT_DIR)
model.eval()

instruction = train_rows[0]['instruction']
user = train_rows[0]['input']
prompt = tokenizer.apply_chat_template([
    {'role': 'system', 'content': instruction},
    {'role': 'user', 'content': user},
], tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=True))

In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive('/content/bhagavata-morph-lora', 'zip', OUTPUT_DIR)
files.download(zip_path)